<a href="https://colab.research.google.com/github/BhavyeNijhawan/swarkavach/blob/main/notebooks/03_asr_ner_intent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03. ASR, entities on noisy transcripts, transformer intent

Three things, in order.

1. Whisper on the generated corpus audio, word error rate against the gold
   scripts. The corpus is scripted, so a perfect reference transcript exists
   for free, which is a luxury no real call recording gives you.
2. The CRF and BiLSTM entity taggers re-run on those Whisper transcripts.
   **This is the centrepiece.** Everything upstream of the text branch assumes
   a transcript, and the gap between entity F1 on gold text and entity F1 on
   ASR text is the honest measurement of what that assumption costs.
3. MuRIL and multilingual DistilBERT fine-tuned for scam intent, on a
   speaker-disjoint split, against the TF-IDF and rule baselines the laptop
   already runs.

**Needs:** a GPU. Whisper small on CPU is roughly 12x slower and the
transformer fine-tunes are impractical. Without a GPU the notebook drops to
Whisper tiny on a 60 call sample, skips the transformers and says so.

**Time:** 25 to 45 minutes on a T4 with the default caps.

**Writes back:**

- `data/results/ner_results.json`, gold against ASR, CRF against BiLSTM
- `data/results/intent_results.json`, transformers against the baselines
- `data/results/asr_results.json`, WER by model and by language setting
- `data/results/intent_branch_scores.json`, one intent score per call for
  notebook 04
- `data/results/attention_*.png`, the attention plots for the report

In [ ]:
# ---------------------------------------------------------------------------
# Get the repository and install the Colab stack.
# Re-running this cell is safe: if the clone is already on disk it pulls.
# ---------------------------------------------------------------------------

# CHANGE THIS to your own fork before you run anything.
REPO_URL = "https://github.com/BhavyeNijhawan/svarkavach.git"
REPO_DIR = "swarkavach"
BRANCH = "main"

import os
import sys
from pathlib import Path

BASE = Path("/content") if Path("/content").exists() else Path.cwd()
REPO = BASE / REPO_DIR

if (REPO / "pyproject.toml").exists():
    print(f"{REPO} is already cloned, pulling instead")
    !git -C "{REPO}" pull --ff-only
else:
    !git clone --branch {BRANCH} {REPO_URL} "{REPO}"

os.chdir(REPO)
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))
print("cwd:", os.getcwd())

!pip install -q -r requirements-colab.txt
!pip install -q -e .

import swarkavach
print("swarkavach", swarkavach.__version__, "ready")

In [ ]:
import shutil

import numpy as np
import torch

from swarkavach import config

config.ensure_dirs()          # data/results must exist before the first savefig

HAS_GPU = torch.cuda.is_available()
DEVICE = torch.device("cuda" if HAS_GPU else "cpu")
GPU = torch.cuda.get_device_name(0) if HAS_GPU else "none"
VRAM_GB = (torch.cuda.get_device_properties(0).total_memory / 1e9) if HAS_GPU else 0.0

if HAS_GPU:
    WHISPER_SIZES = ["small"]
    if VRAM_GB >= 14:
        WHISPER_SIZES.append("medium")     # medium needs about 5 GB, plus headroom
    MAX_CALLS_ASR = 200
    BERT_EPOCHS = 4
    BERT_BATCH = 32
    DO_TRANSFORMERS = True
else:
    WHISPER_SIZES = ["tiny"]
    MAX_CALLS_ASR = 60
    BERT_EPOCHS = 1
    BERT_BATCH = 8
    DO_TRANSFORMERS = False
    print("NO GPU. Whisper tiny on 60 calls, transformer fine-tuning skipped.")
    print("Every exported number carries reduced=True.")

REDUCED = not HAS_GPU
print(f"device {DEVICE}  gpu {GPU} ({VRAM_GB:.1f} GB)  torch {torch.__version__}")
print(f"whisper sizes to run: {WHISPER_SIZES}")
print(f"free disk {shutil.disk_usage('.').free / 1e9:.1f} GB")

## The corpus

If notebook 01 has not run in this session there is no corpus on disk, so
generate one. `--audio` renders every call through the repository's
source-filter synthesiser, which is what Whisper is going to hear.

In [ ]:
import subprocess
import sys
from collections import Counter
from pathlib import Path

calls_dir = Path("data/corpus/calls")
have_corpus = calls_dir.exists() and any(calls_dir.glob("*.json"))
if not have_corpus:
    print("no corpus on disk, generating one")
    proc = subprocess.run([sys.executable, "-m", "swarkavach.cli", "gen-corpus",
                           "--n", "480", "--audio"], capture_output=True, text=True)
    print((proc.stdout or "")[-2000:])
    if proc.returncode != 0:
        print((proc.stderr or "")[-2000:])

from swarkavach.corpus.generator import load_corpus

CALLS = load_corpus()
WITH_AUDIO = [c for c in CALLS if c.audio_path and Path(c.audio_path).exists()]
print(f"{len(CALLS)} calls, {len(WITH_AUDIO)} with audio")
print("splits:", Counter(c.split for c in CALLS))
print("cells:", Counter(c.cell for c in CALLS))
print("speakers:", len({c.speaker_id for c in CALLS}))

TRAIN = [c for c in CALLS if c.split == "train"]
DEV = [c for c in CALLS if c.split == "dev"]
TEST = [c for c in CALLS if c.split == "test"]

# The splits are meant to be speaker-disjoint. Check, do not assume.
sp = {"train": {c.speaker_id for c in TRAIN}, "dev": {c.speaker_id for c in DEV},
      "test": {c.speaker_id for c in TEST}}
for a in ("train", "dev", "test"):
    for b in ("train", "dev", "test"):
        if a < b:
            n = len(sp[a] & sp[b])
            print(f"speaker overlap {a}/{b}: {n} {'ok' if n == 0 else 'LEAK'}")

## Whisper, and the script problem

The single largest source of word error rate on Hinglish is not
mispronunciation, it is orthography. Ask Whisper for Hindi and it returns
Devanagari. The corpus gold is romanised Hinglish, so a perfect Devanagari
transcription scores near 100 percent WER against it, and the number tells you
nothing about whether the words were heard correctly.

So both settings are run:

- `language="en"` forces Latin output, which is the same script as the gold
  and gives a WER that means something. Whisper transliterates Hindi words
  inconsistently under this setting, which is itself part of what is being
  measured.
- `language="hi"` is what a naive pipeline would use, and its WER against
  romanised gold is reported so the report can name the trap rather than
  fall into it.

The `en` transcripts are what the entity taggers read below, because a tagger
trained on romanised tokens cannot do anything with Devanagari.

Turns are transcribed individually using the corpus turn boundaries, not the
whole call in one pass. Turn-aligned ASR text is what the entity comparison
needs, and cutting on known boundaries removes segmentation error from a
measurement that is about recognition.

In [ ]:
import json
import time
from pathlib import Path

import numpy as np

from swarkavach.audioio import read_audio, resample
from swarkavach.config import SETTINGS, TARGET_SR, WIDEBAND_SR
from swarkavach.schema import tokenize

# WER comes from the project's own module so the number here and the number a
# local run prints are the same number. aggregate_wer is total edits over total
# reference words, which is what ASR papers report; averaging per-utterance
# rates would let a three word turn outweigh a forty word one.
#
# The hinglish flag runs the romanisation canonicaliser from text/normalize.py
# over both sides first. Plan B section 2.3 predicts that recovers several
# points, and running it both ways measures rather than assumes that.
try:
    from swarkavach.text.asr import aggregate_wer, wer_details, word_error_rate

    WER_SOURCE = "swarkavach.text.asr"
except Exception as exc:
    print(f"swarkavach.text.asr not importable ({type(exc).__name__}), "
          f"using a local edit distance instead")

    def word_error_rate(ref: str, hyp: str, hinglish: bool = False) -> float:
        r = [t.lower() for t in tokenize(ref) if t.isalnum()]
        h = [t.lower() for t in tokenize(hyp) if t.isalnum()]
        if not r:
            return 0.0 if not h else 1.0
        prev = list(range(len(h) + 1))
        for i, a in enumerate(r, 1):
            cur = [i] + [0] * len(h)
            for j, b in enumerate(h, 1):
                cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (a != b))
            prev = cur
        return float(prev[-1]) / float(len(r))

    def aggregate_wer(refs, hyps, hinglish: bool = False):
        vals = [word_error_rate(r, h, hinglish) for r, h in zip(refs, hyps)]
        return {"wer": float(np.mean(vals)) if vals else 0.0,
                "mean_utterance_wer": float(np.mean(vals)) if vals else 0.0,
                "n_utterances": len(vals), "n_ref_words": 0}

    WER_SOURCE = "notebook fallback"

print(f"WER implementation: {WER_SOURCE}")

ASR_CALLS = WITH_AUDIO[:MAX_CALLS_ASR]
print(f"transcribing {len(ASR_CALLS)} calls")

TRANSCRIPTS = {}          # (model, lang) -> {call_id: {turn_index: text}}
ASR_TIMING = {}


def turn_clip(x, sr, turn, pad_s: float = 0.10):
    a = max(0, int((turn.t_start - pad_s) * sr))
    b = min(len(x), int((turn.t_end + pad_s) * sr))
    if b - a < int(0.2 * sr):
        b = min(len(x), a + int(0.2 * sr))
    return x[a:b]


try:
    import whisper
except Exception as exc:
    whisper = None
    print(f"whisper not importable: {type(exc).__name__}: {exc}")

if whisper is not None:
    for size in WHISPER_SIZES:
        print(f"\nloading whisper {size}")
        model = whisper.load_model(size, device=DEVICE)
        for lang in ("en", "hi"):
            key = (size, lang)
            TRANSCRIPTS[key] = {}
            t0 = time.time()
            for k, call in enumerate(ASR_CALLS):
                try:
                    x, sr = read_audio(call.audio_path, sr=TARGET_SR)
                    x16 = resample(np.asarray(x, dtype=np.float32), sr, WIDEBAND_SR)
                except Exception:
                    continue
                per_turn = {}
                for turn in call.turns:
                    clip = turn_clip(x16, WIDEBAND_SR, turn)
                    try:
                        out = model.transcribe(
                            np.asarray(clip, dtype=np.float32), language=lang,
                            task="transcribe", fp16=HAS_GPU, verbose=False,
                            condition_on_previous_text=False)
                        per_turn[turn.index] = (out.get("text") or "").strip()
                    except Exception:
                        per_turn[turn.index] = ""
                TRANSCRIPTS[key][call.call_id] = per_turn
                if (k + 1) % 25 == 0:
                    print(f"  {size}/{lang}: {k + 1}/{len(ASR_CALLS)} calls "
                          f"({time.time() - t0:.0f}s)", flush=True)
            ASR_TIMING[f"{size}/{lang}"] = round(time.time() - t0, 1)
            print(f"  {size}/{lang} done in {time.time() - t0:.0f}s")
        del model
        if HAS_GPU:
            torch.cuda.empty_cache()
else:
    print("no whisper, the ASR condition below will be marked as not run")

In [ ]:
import numpy as np
import pandas as pd

WER_TABLE = {}
by_call = {c.call_id: c for c in ASR_CALLS}

rows = []
for (size, lang), per_call in TRANSCRIPTS.items():
    refs, hyps = [], []
    for cid, per_turn in per_call.items():
        for turn in by_call[cid].turns:
            refs.append(turn.text)
            hyps.append(per_turn.get(turn.index, ""))
    key = f"{size}/{lang}"
    plain = aggregate_wer(refs, hyps, hinglish=False)
    canon = aggregate_wer(refs, hyps, hinglish=True)
    WER_TABLE[key] = {
        "wer": round(plain["wer"], 4),
        "wer_hinglish_normalised": round(canon["wer"], 4),
        "mean_utterance_wer": round(plain["mean_utterance_wer"], 4),
        "n_utterances": plain["n_utterances"],
        "n_ref_words": plain["n_ref_words"],
        "n_calls": len(per_call),
        "seconds": ASR_TIMING.get(key),
    }
    rows.append({"model": size, "language": lang,
                 "WER": WER_TABLE[key]["wer"],
                 "WER (canonicalised)": WER_TABLE[key]["wer_hinglish_normalised"],
                 "turns": plain["n_utterances"],
                 "seconds": ASR_TIMING.get(key)})

if rows:
    print(pd.DataFrame(rows).to_string(index=False))
    print("\nThe canonicalised column applies text/normalize.py to both sides "
          "before scoring. The gap between the two columns is how much of the "
          "error was spelling rather than recognition.")
    print("\nA sample, so the numbers above have a face:")
    key = next(k for k in TRANSCRIPTS if k[1] == "en")
    cid = next(iter(TRANSCRIPTS[key]))
    call = by_call[cid]
    for turn in call.turns[:4]:
        print(f"  gold : {turn.text}")
        print(f"  {key[0]}/{key[1]} : {TRANSCRIPTS[key][cid].get(turn.index, '')}")
        print()
else:
    print("no transcripts produced")

# The best Latin-script transcript is what the entity taggers will read.
BEST_ASR = None
lat = [k for k in TRANSCRIPTS if k[1] == "en"]
if lat:
    BEST_ASR = min(lat, key=lambda k: WER_TABLE[f"{k[0]}/{k[1]}"]["wer"])
    print(f"entity taggers will read: {BEST_ASR[0]}/{BEST_ASR[1]}")

## Entity F1: gold transcript against ASR transcript

The taggers are trained once, on the gold text of the training split, and then
asked the same question twice: once on gold test transcripts and once on
Whisper's version of the same audio. Training on gold and testing on ASR is
the realistic setup, because gold transcripts exist at training time (you
scripted the corpus) and do not exist at inference time.

**How the two conditions are scored the same way.** ASR changes the token
sequence, so an exact-span BIO comparison against the gold token indices is
undefined for the ASR condition. Instead both conditions are scored as entity
extraction: per turn, the multiset of (entity type, normalised entity text)
that the tagger produced, against the multiset the gold BIO contains. An
entity Whisper deleted from the audio simply never appears in the prediction
and counts as a miss, which is the correct accounting: the pipeline really did
lose it.

The exact-span `evaluate.entity_prf` number is reported alongside for the gold
condition, so this table can be lined up against the local run's numbers.

In [ ]:
import re
from collections import Counter
from typing import Dict, List

import numpy as np

from swarkavach.evaluate import entity_prf
from swarkavach.schema import decode_bio, tokenize
from swarkavach.text.ner_bilstm import BiLSTMTagger, majority_baseline, most_frequent_tag_baseline
from swarkavach.text.ner_crf import CRFTagger

print("training the CRF tagger on the gold training split")
crf = CRFTagger()
crf.fit(TRAIN)

print("training the BiLSTM tagger")
bilstm = BiLSTMTagger(epochs=40, verbose=True)
bilstm.fit(TRAIN)
print("bilstm history:", bilstm.history_)

TAGGERS = {"crf": crf, "bilstm": bilstm}


def norm(s: str) -> str:
    return " ".join(t.lower() for t in tokenize(s))


def entity_multiset(tokens, bio) -> Counter:
    """(type, normalised surface) counts for one turn."""
    return Counter((e.type, norm(e.text)) for e in decode_bio(list(tokens), list(bio)))


def extraction_prf(gold_sets, pred_sets):
    """Entity extraction P/R/F1 over per-turn multisets, plus per type."""
    tp = fp = fn = 0
    per_type: Dict[str, List[int]] = {}
    for g, p in zip(gold_sets, pred_sets):
        for key in set(g) | set(p):
            hit = min(g.get(key, 0), p.get(key, 0))
            over = max(p.get(key, 0) - g.get(key, 0), 0)
            under = max(g.get(key, 0) - p.get(key, 0), 0)
            tp += hit; fp += over; fn += under
            row = per_type.setdefault(key[0], [0, 0, 0])
            row[0] += hit; row[1] += over; row[2] += under
    def prf(t, f, n):
        p = t / (t + f) if t + f else 0.0
        r = t / (t + n) if t + n else 0.0
        return {"p": round(p, 4), "r": round(r, 4),
                "f1": round(2 * p * r / (p + r), 4) if p + r else 0.0,
                "support": t + n, "predicted": t + f, "correct": t}
    out = prf(tp, fp, fn)
    out["per_type"] = {k: prf(*v) for k, v in sorted(per_type.items())}
    return out


ASR_TEST = [c for c in TEST if BEST_ASR and c.call_id in TRANSCRIPTS.get(BEST_ASR, {})]
print(f"\n{len(TEST)} test calls, {len(ASR_TEST)} of them transcribed by Whisper")

NER = {"overall": {}, "per_type": {}, "exact_span": {}}
EVAL_CALLS = ASR_TEST if ASR_TEST else TEST

for name, tagger in TAGGERS.items():
    NER["overall"][name] = {}
    NER["per_type"][name] = {}

    # --- gold transcripts -------------------------------------------------
    turns = [t for c in EVAL_CALLS for t in c.turns if t.tokens]
    gold_sets = [entity_multiset(t.tokens, t.bio) for t in turns]
    preds = tagger.predict_batch([t.tokens for t in turns], [t.lang for t in turns])
    preds = [p if len(p) == len(t.tokens) else ["O"] * len(t.tokens)
             for p, t in zip(preds, turns)]
    pred_sets = [entity_multiset(t.tokens, p) for t, p in zip(turns, preds)]
    r = extraction_prf(gold_sets, pred_sets)
    NER["overall"][name]["gold"] = {k: r[k] for k in ("p", "r", "f1", "support")}
    NER["per_type"][name]["gold"] = r["per_type"]

    # the exact-span number, so this is comparable to the local evaluation
    ex = entity_prf([list(t.bio) for t in turns], preds, [list(t.tokens) for t in turns])
    NER["exact_span"][name] = {"p": round(ex["p"], 4), "r": round(ex["r"], 4),
                               "f1": round(ex["f1"], 4), "support": ex["support"]}

    # --- ASR transcripts --------------------------------------------------
    if ASR_TEST and BEST_ASR:
        g_sets, p_sets = [], []
        for c in ASR_TEST:
            per_turn = TRANSCRIPTS[BEST_ASR][c.call_id]
            for t in c.turns:
                if not t.tokens:
                    continue
                hyp_tokens = tokenize(per_turn.get(t.index, ""))
                g_sets.append(entity_multiset(t.tokens, t.bio))
                if not hyp_tokens:
                    p_sets.append(Counter())
                    continue
                bio = tagger.predict_turn(hyp_tokens)
                if len(bio) != len(hyp_tokens):
                    bio = ["O"] * len(hyp_tokens)
                p_sets.append(entity_multiset(hyp_tokens, bio))
        r = extraction_prf(g_sets, p_sets)
        NER["overall"][name]["asr"] = {k: r[k] for k in ("p", "r", "f1", "support")}
        NER["per_type"][name]["asr"] = r["per_type"]

    got = NER["overall"][name]
    line = f"{name:<7} gold F1 {got['gold']['f1']:.3f}"
    if "asr" in got:
        drop = got["gold"]["f1"] - got["asr"]["f1"]
        line += (f"   ASR F1 {got['asr']['f1']:.3f}   "
                 f"drop {drop:.3f} ({100 * drop / max(got['gold']['f1'], 1e-9):.0f}%)")
    print(line)

BASELINES = {
    "majority": majority_baseline(EVAL_CALLS),
    "most_frequent_tag": most_frequent_tag_baseline(TRAIN, EVAL_CALLS),
}
for k, v in BASELINES.items():
    print(f"{k:<18} exact-span F1 {v['f1']:.3f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = list(NER["overall"])
srcs = [s for s in ("gold", "asr") if any(s in NER["overall"][m] for m in models)]
if models and srcs:
    x = np.arange(len(models))
    w = 0.36
    fig, ax = plt.subplots(figsize=(6.4, 3.6))
    for i, s in enumerate(srcs):
        vals = [NER["overall"][m].get(s, {}).get("f1", 0.0) for m in models]
        bars = ax.bar(x + (i - (len(srcs) - 1) / 2) * w, vals, w,
                      label="gold transcript" if s == "gold" else "Whisper transcript")
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, v + 0.015, f"{v:.2f}",
                    ha="center", fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels([m.upper() for m in models])
    ax.set_ylim(0, 1.05); ax.set_ylabel("entity F1 (extraction)")
    ax.set_title("What a noisy transcript costs the entity taggers")
    ax.legend(); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("data/results/ner_gold_vs_asr.png", dpi=140)
    plt.show()

    print("\nPer entity type, gold against ASR:")
    for m in models:
        g = NER["per_type"][m].get("gold", {})
        a = NER["per_type"][m].get("asr", {})
        print(f"\n{m.upper()}")
        print(f"  {'type':<20} {'gold F1':>8} {'ASR F1':>8} {'drop':>7}")
        for t in sorted(set(g) | set(a)):
            gf = g.get(t, {}).get("f1", 0.0)
            af = a.get(t, {}).get("f1", 0.0)
            print(f"  {t:<20} {gf:>8.3f} {af:>8.3f} {gf - af:>7.3f}")

## Scam intent: MuRIL and DistilBERT against the baselines

MuRIL (`google/muril-base-cased`) is trained on seventeen Indian languages
including transliterated text, which is the reason to prefer it here over a
general multilingual model: romanised Hindi is in its pretraining data rather
than being an accident of byte-pair encoding. Multilingual DistilBERT is the
control, at roughly half the parameters.

**Labels.** The corpus labels calls, not turns, so a caller turn inherits its
call's label. That is weak supervision and it is worth saying in the report:
the greeting at the start of a scam call is labelled scam. Call-level scores
are formed the same way `fusion.featurize.build_features` does it, `0.6 * max
+ 0.4 * mean` over caller turns, so the transformer and the baselines are
compared on identical pooling.

**Split.** By speaker, not by call. The generated corpus reuses a pool of
speakers across calls, and a model that has seen a speaker's phrasing in
training will recognise it in test for reasons that have nothing to do with
fraud. The corpus already splits this way; the cell verifies it.

In [ ]:
import time

import numpy as np

INTENT = {"models": {}}
CALL_SCORES = {}          # model name -> {call_id: score}


def call_score(turn_scores):
    a = np.asarray(turn_scores, dtype=float) if len(turn_scores) else np.zeros(1)
    return float(np.clip(0.6 * a.max() + 0.4 * a.mean(), 0.0, 1.0))


def caller_texts(calls):
    """(texts, labels, call ids) at caller-turn level."""
    texts, labels, cids = [], [], []
    for c in calls:
        for t in c.caller_turns():
            if not t.text.strip():
                continue
            texts.append(t.text)
            labels.append(int(c.label_scam))
            cids.append(c.call_id)
    return texts, labels, cids


TR_X, TR_Y, _ = caller_texts(TRAIN)
TE_X, TE_Y, TE_ID = caller_texts(TEST)
print(f"{len(TR_X)} training turns, {len(TE_X)} test turns, "
      f"scam share train {np.mean(TR_Y):.2f} test {np.mean(TE_Y):.2f}")

MODELS_TO_TUNE = []
if DO_TRANSFORMERS:
    MODELS_TO_TUNE = [("muril", "google/muril-base-cased"),
                      ("distilbert_multi", "distilbert-base-multilingual-cased")]
else:
    print("transformer fine-tuning skipped, no GPU")

FINETUNED = {}

for short, hf_name in MODELS_TO_TUNE:
    print(f"\n=== {short}  ({hf_name}) ===")
    try:
        from torch.utils.data import DataLoader, TensorDataset
        from transformers import AutoModelForSequenceClassification, AutoTokenizer

        tok = AutoTokenizer.from_pretrained(hf_name)
        model = AutoModelForSequenceClassification.from_pretrained(
            hf_name, num_labels=2).to(DEVICE)

        def encode(texts, labels):
            enc = tok(list(texts), truncation=True, max_length=96,
                      padding="max_length", return_tensors="pt")
            return TensorDataset(enc["input_ids"], enc["attention_mask"],
                                 torch.tensor(labels, dtype=torch.long))

        tr_loader = DataLoader(encode(TR_X, TR_Y), batch_size=BERT_BATCH, shuffle=True)
        te_loader = DataLoader(encode(TE_X, TE_Y), batch_size=BERT_BATCH * 2)

        opt = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
        steps = max(1, len(tr_loader) * BERT_EPOCHS)
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-5, total_steps=steps)

        t0 = time.time()
        for epoch in range(BERT_EPOCHS):
            model.train()
            losses = []
            for ids, mask, y in tr_loader:
                ids, mask, y = ids.to(DEVICE), mask.to(DEVICE), y.to(DEVICE)
                opt.zero_grad(set_to_none=True)
                out = model(input_ids=ids, attention_mask=mask, labels=y)
                out.loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step(); sched.step()
                losses.append(float(out.loss.item()))
            print(f"  epoch {epoch + 1}  loss {np.mean(losses):.4f}  "
                  f"{time.time() - t0:.0f}s")

        model.eval()
        probs = []
        with torch.no_grad():
            for ids, mask, _ in te_loader:
                logits = model(input_ids=ids.to(DEVICE), attention_mask=mask.to(DEVICE)).logits
                probs.append(torch.softmax(logits, dim=1)[:, 1].float().cpu().numpy())
        probs = np.concatenate(probs)

        per_call = {}
        for cid, p in zip(TE_ID, probs):
            per_call.setdefault(cid, []).append(float(p))
        CALL_SCORES[short] = {cid: call_score(v) for cid, v in per_call.items()}
        FINETUNED[short] = (model, tok)
        print(f"  fine-tuned in {time.time() - t0:.0f}s, "
              f"{len(CALL_SCORES[short])} calls scored")
    except Exception as exc:
        print(f"  {short} failed: {type(exc).__name__}: {exc}")

In [ ]:
import numpy as np

from swarkavach.text.intent_rules import RuleIntent
from swarkavach.text.intent_tfidf import TfidfIntent

# The two baselines the laptop already runs, refitted on the same split so the
# comparison is like for like.
rules = RuleIntent()
CALL_SCORES["rules"] = {
    c.call_id: call_score([rules.score_turn(t.tokens) for t in c.caller_turns()])
    for c in TEST}

tfidf = TfidfIntent()
tfidf.fit(TRAIN)
CALL_SCORES["tfidf"] = {
    c.call_id: call_score([tfidf.score_turn(t.tokens) for t in c.caller_turns()])
    for c in TEST}

from swarkavach.evaluate import binary_scores, roc_auc

y_by_call = {c.call_id: int(c.label_scam) for c in TEST}
rows = []
for name, scores in CALL_SCORES.items():
    ids = sorted(scores)
    s = np.asarray([scores[i] for i in ids])
    y = np.asarray([y_by_call[i] for i in ids])
    if len(set(y.tolist())) < 2:
        continue
    b = binary_scores(s, y, 0.5)
    INTENT["models"][name] = {
        "auc": round(roc_auc(s, y), 4),
        "accuracy": round(b["accuracy"], 4),
        "f1": round(b["f1"], 4),
        "precision": round(b["precision"], 4),
        "recall": round(b["recall"], 4),
        "confusion": {k: int(b[k]) for k in ("tp", "fp", "fn", "tn")},
        "n": len(ids),
    }
    rows.append((name, INTENT["models"][name]))

print(f"{'model':<18} {'AUC':>7} {'acc':>7} {'F1':>7} {'P':>7} {'R':>7}")
for name, m in sorted(rows, key=lambda r: -r[1]["auc"]):
    print(f"{name:<18} {m['auc']:>7.3f} {m['accuracy']:>7.3f} {m['f1']:>7.3f} "
          f"{m['precision']:>7.3f} {m['recall']:>7.3f}")

## Attention

Module 4 of the syllabus is attention, so the report needs a picture of it
rather than a citation to it. What is plotted is the last encoder layer,
averaged over heads, taking the row for the `[CLS]` position: how much each
word token the pooled sentence representation drew from. Sub-word pieces are
merged back into words by summing.

Attention weights are not an explanation of the decision, and the report
should not claim they are. They show where the representation looked, which is
useful and much weaker than a causal attribution. The exact Shapley values in
the fusion layer are what carry the explanation claim in this project.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ATTENTION = {}

# Two scam turns with real extraction pressure in them, plus one benign turn
# as the control.
def pick_examples():
    out = []
    for c in TEST:
        if c.label_scam:
            for t in c.caller_turns():
                if any(b != "O" for b in t.bio):
                    out.append(("scam", t.text))
                    break
        if len(out) >= 2:
            break
    for c in TEST:
        if not c.label_scam and c.caller_turns():
            out.append(("benign", c.caller_turns()[0].text))
            break
    return out


EXAMPLES = pick_examples()
for short, (model, tok) in FINETUNED.items():
    for j, (kind, text) in enumerate(EXAMPLES):
        try:
            enc = tok(text, return_tensors="pt", truncation=True, max_length=96)
            with torch.no_grad():
                out = model(**{k: v.to(DEVICE) for k, v in enc.items()},
                            output_attentions=True)
            att = out.attentions[-1][0].mean(dim=0)[0].float().cpu().numpy()  # CLS row
            pieces = tok.convert_ids_to_tokens(enc["input_ids"][0])

            # merge sub-words back into words
            words, weights = [], []
            for piece, w in zip(pieces, att):
                if piece in tok.all_special_tokens:
                    continue
                if piece.startswith("##") and words:
                    words[-1] += piece[2:]
                    weights[-1] += float(w)
                else:
                    words.append(piece)
                    weights.append(float(w))
            weights = np.asarray(weights)
            if weights.sum() > 0:
                weights = weights / weights.sum()

            ATTENTION.setdefault(short, []).append(
                {"kind": kind, "text": text, "tokens": words,
                 "weights": [round(float(w), 5) for w in weights]})

            fig, ax = plt.subplots(figsize=(max(6, 0.42 * len(words)), 2.6))
            colors = ["tab:red" if kind == "scam" else "tab:blue"] * len(words)
            ax.bar(range(len(words)), weights, color=colors, alpha=0.85)
            ax.set_xticks(range(len(words)))
            ax.set_xticklabels(words, rotation=60, ha="right", fontsize=8)
            ax.set_ylabel("CLS attention")
            ax.set_title(f"{short}, last layer, head-averaged  [{kind}]", fontsize=10)
            ax.grid(axis="y", alpha=0.3)
            plt.tight_layout()
            path = f"data/results/attention_{short}_{j}.png"
            plt.savefig(path, dpi=140)
            plt.show()
            print(f"saved {path}")
            top = sorted(zip(words, weights), key=lambda p: -p[1])[:6]
            print("  top tokens:", ", ".join(f"{w} {v:.3f}" for w, v in top))
        except Exception as exc:
            print(f"  attention plot failed for {short}: {type(exc).__name__}: {exc}")

if not FINETUNED:
    print("no fine-tuned transformer, nothing to visualise")

## Export

In [ ]:
# ---------------------------------------------------------------------------
# Export helpers. Each notebook carries its own copy so it can run on its own.
#
# Everything written here lands in data/results/ with "source": "colab", a UTC
# timestamp and the GPU name, which is what the local console's provenance
# table reads to tell your laptop numbers apart from your Colab numbers.
# ---------------------------------------------------------------------------
import json
import platform
import zipfile
from datetime import datetime, timezone
from pathlib import Path

NOTEBOOK = "03_asr_ner_intent"
RESULTS = Path("data/results")
RESULTS.mkdir(parents=True, exist_ok=True)


def gpu_name() -> str:
    try:
        import torch

        if torch.cuda.is_available():
            return torch.cuda.get_device_name(0)
    except Exception:
        pass
    return "none (CPU runtime)"


def utc_now() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")


def colab_stamp(notes: str = "") -> dict:
    try:
        import torch

        tv = str(torch.__version__)
    except Exception:
        tv = "not installed"
    return {
        "source": "colab",
        "generated": utc_now(),
        "gpu": gpu_name(),
        "torch": tv,
        "python": platform.python_version(),
        "notebook": NOTEBOOK,
        "notes": notes,
    }


WRITTEN = []


def write_result(name: str, obj: dict, notes: str = "") -> Path:
    """Write one result file, stamped, and remember it for the zip."""
    doc = dict(obj)
    doc.update(colab_stamp(notes))
    path = RESULTS / name
    path.write_text(json.dumps(doc, indent=2, default=float), encoding="utf-8")
    if name not in WRITTEN:
        WRITTEN.append(name)
    print(f"wrote {path}  ({path.stat().st_size / 1024:.1f} kB)")
    return path


def update_provenance(entries) -> Path:
    """Merge these Colab entries into data/results/provenance.json.

    Entries for other artifacts are kept, so a local `swarkavach evaluate` run
    and a Colab run can sit in the same table. Unzip the Colab results AFTER
    the local evaluate, otherwise the local file overwrites this one.
    """
    path = RESULTS / "provenance.json"
    doc = {"entries": []}
    if path.exists():
        try:
            doc = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            pass
    new_names = {e["artifact"] for e in entries}
    kept = [e for e in doc.get("entries", []) if e.get("artifact") not in new_names]
    doc["entries"] = kept + list(entries)
    doc["generated"] = utc_now()
    doc["host"] = platform.platform()
    doc["gpu"] = gpu_name()
    path.write_text(json.dumps(doc, indent=2), encoding="utf-8")
    if "provenance.json" not in WRITTEN:
        WRITTEN.append("provenance.json")
    print(f"provenance now lists {len(doc['entries'])} artifacts")
    return path


def export_zip(extra_paths=(), zip_name: str = None) -> Path:
    """Zip the result files plus anything extra, then hand it to the browser."""
    out = Path(zip_name or f"swarkavach_{NOTEBOOK}.zip")
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for name in WRITTEN:
            f = RESULTS / name
            if f.exists():
                z.write(f, arcname=f"data/results/{f.name}")
        for extra in extra_paths:
            p = Path(extra)
            if p.exists():
                z.write(p, arcname=p.as_posix())
            else:
                print(f"  skipped missing {p}")

    names = zipfile.ZipFile(out).namelist()
    print(f"\n{out.name}  {out.stat().st_size / 1024:.1f} kB")
    for n in names:
        print("   ", n)

    try:
        from google.colab import files

        files.download(str(out))
        print("\nDownload started. Unzip it over your local project root.")
    except Exception as exc:
        print(f"\nnot running on Colab ({type(exc).__name__}), zip is at {out.resolve()}")
    return out

In [ ]:
from pathlib import Path

transcripts_present = ["gold"] + (["asr"] if BEST_ASR else [])
ASR_LABEL = f"{BEST_ASR[0]}/{BEST_ASR[1]}" if BEST_ASR else "no"

ner_doc = {
    "models": list(NER["overall"]),
    "transcripts": transcripts_present,
    "overall": NER["overall"],
    "per_type": NER["per_type"],
    "exact_span_gold": NER["exact_span"],
    "baselines": {k: {"f1": round(v["f1"], 4), "p": round(v["precision"], 4),
                      "r": round(v["recall"], 4)} for k, v in BASELINES.items()},
    "n_turns": sum(len([t for t in c.turns if t.tokens]) for c in EVAL_CALLS),
    "n_calls": len(EVAL_CALLS),
    "asr_model": ASR_LABEL if BEST_ASR else None,
    "scoring": ("entity extraction: per-turn multiset of (type, normalised surface). "
                "Exact-span entity_prf is reported separately for the gold condition "
                "so it lines up with the local evaluation."),
    "reduced": REDUCED,
}
write_result("ner_results.json", ner_doc,
             notes="CRF and BiLSTM, gold transcripts against Whisper transcripts")

intent_doc = dict(INTENT)
intent_doc.update({
    "split": "speaker-disjoint test split of the generated corpus",
    "pooling": "0.6 * max caller turn + 0.4 * mean, same as fusion.featurize",
    "label_note": "call-level labels applied to caller turns, weak supervision",
    "epochs": BERT_EPOCHS,
    "attention": ATTENTION,
    "reduced": REDUCED,
    "n_test_calls": len(TEST),
})
write_result("intent_results.json", intent_doc,
             notes="MuRIL and multilingual DistilBERT against TF-IDF and rules")

write_result("asr_results.json", {
    "wer": WER_TABLE,
    "implementation": WER_SOURCE,
    "n_calls": len(ASR_CALLS),
    "note": ("language='hi' returns Devanagari and the corpus gold is romanised, "
             "so that row's WER measures a script mismatch and not recognition. "
             "The entity comparison uses the language='en' transcripts."),
}, notes="Whisper WER against the gold corpus scripts")

best_intent = max(CALL_SCORES, key=lambda k: INTENT["models"].get(k, {}).get("auc", 0)) \
    if INTENT["models"] else None
if best_intent:
    write_result("intent_branch_scores.json", {
        "scores": CALL_SCORES[best_intent],
        "model": best_intent,
        "auc": INTENT["models"][best_intent]["auc"],
        "all_models": {k: v for k, v in CALL_SCORES.items()},
    }, notes=f"per-call intent score from {best_intent}, for notebook 04")

update_provenance([
    {"artifact": "ner_results.json", "source": "colab", "generated": utc_now(),
     "notes": (f"CRF and BiLSTM on {len(EVAL_CALLS)} test calls, gold against "
               f"{ASR_LABEL} Whisper transcripts")},
    {"artifact": "intent_results.json", "source": "colab", "generated": utc_now(),
     "notes": (f"{', '.join(INTENT['models'])} on {gpu_name()}"
               + (", transformers skipped without a GPU" if REDUCED else ""))},
    {"artifact": "asr_results.json", "source": "colab", "generated": utc_now(),
     "notes": f"Whisper {', '.join(WHISPER_SIZES)} on {len(ASR_CALLS)} calls"},
])

export_zip(extra_paths=sorted(Path("data/results").glob("attention_*.png"))
           + [Path("data/results/ner_gold_vs_asr.png")])

## Next

`04_fusion_and_eval.ipynb`. It reads `antispoof_branch_scores.json` from
notebook 02 and `intent_branch_scores.json` from this one, rebuilds the
25-feature vectors with those real-data branch scores in place, and runs the
ablation that the whole project is built to test.